# Burnout Detection — Replication Package

**What this notebook does, top to bottom:**
1. **Setup / Helpers**: mount Drive, load a developer's raw activity files.
2. **Signal cells**: (Arousal, Commits, Pronouns, Lexical Diversity, Lexical
   Features, Off-Hours, PR Throughput, Task Abandonment, Sentiment), each
   cell computes its monthly signal and writes it to the developer's
   `{dev_name}_metrics.xlsx` workbook.
3. **Burnout Calculation**: combines the
   signals above into monthly Exhaustion / Disengagement / Combined risk
   scores, detects sustained high-risk onsets, and measures the departure /
   activity-decline / role-withdrawal outcomes that follow each onset,
   scoped to the one developer this notebook runs on.
4. **Main Loop**: prompts for a developer name and runs steps 2-3 for them.

Run cells top to bottom, then answer the prompt in the Main Loop cell.

## SETUP
Installs the packages the rest of the notebook needs. Safe to re-run.

In [ ]:
# SETUP
# =============================================================================
# One-time dependency install for this replication package. torch,
# transformers, and sentence-transformers are not part of the default Colab
# image.
# =============================================================================

%pip install -q openpyxl sentence-transformers transformers torch


## HELPERS
Utility functions shared across all metric modules.

In [ ]:
# HELPERS
# =============================================================================
# Utility functions shared across all metric modules.
#
# find_all_developers     : finds all folders with commit data.
#
# clean_text              : removes HTML, URLs, inline code from text.
#
# load_messages           : loads all text artifacts for a developer into a DataFrame.
#
# compute_metric_series   : builds the value/smoothed/std frame a signal is exported with
#                           (the non-plotting half of what AllMetrics_3-4's
#                           plot_metric_time_series used to hand back).
#
# save_metric_data        : buffers metric DataFrames for later export
#
# flush_metric_data       : writes all buffered data to an Excel file
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
from google.colab import drive
import torch


# PATH TO DEVELOPER FOLDERS
DEVPATH = Path("")
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
excel_cache = {} # dev_name : {sheetname : df }

print(f"Data directory exists: {DEVPATH.exists()}")
print(f"Using device: {DEVICE}")


def find_all_developers(dev_profiles_dir):
    """
    Return sorted list of developer Path objects that have at least one commits file

        Params:
          dev_profiles_dir (Path): Directory with developer subfolders

        Returns:
           List of developer folder paths (sorted alphabetically)

    """
    if not dev_profiles_dir.exists():
        print("Directory not found")
        return []

    developers = []
    for item in dev_profiles_dir.iterdir():
        developers.append(item)

    return sorted(developers, key=lambda x: x.name)


def clean_text(text):
    """
    Clean raw text by removing HTML tags, URLs, and inline code fragments

      Params:
        text (string): Raw text from a given message

      Returns:
        string of cleaned text

    """
    text = re.sub(r"<[^>]+>", " ", text)                # HTML tags
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # URLs
    text = re.sub(r"`[^`]*`", " CODE_FRAGMENT ", text)  # Inline code
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_messages(dev_path):
    """
    Load and merge all text for one developer

      Params:
        dev_path(Path): Path to the developer's folder

      Returns:
        pd.DataFrame(['datetime', 'text']) sorted chronoligcally
    """

    # [Type, filename, date column, text column]
    file_patterns = [
        ("commits",          f"{dev_path.name}_commits.xlsx",          "date",         "message"),
        ("issue_comments",   f"{dev_path.name}_issue_comments.xlsx",    "created_at",   "comment_body"),
        ("prs",              f"{dev_path.name}_prs.xlsx",               "created_at",   "title"),
        ("review_comments",  f"{dev_path.name}_review_comments.xlsx",   "created_at",   "comment_body"),
        ("reviews",          f"{dev_path.name}_reviews.xlsx",           "submitted_at", "pr_title"),
    ]

    dataframes = []
    for _, filename, date_col, text_col in file_patterns:
        file_path = dev_path / filename

        df = pd.read_excel(file_path)
        if date_col not in df.columns:
            print(f"  Date column '{date_col}' not found in {filename}")
            continue

        # convert date column to datetime
        df["datetime"] = pd.to_datetime(df[date_col], errors="coerce")
        df = df.dropna(subset=["datetime"])

        # possible alternate text columns
        if text_col not in df.columns:
            fallbacks = ["body", "comment_body", "comment", "text", "message", "title", "pr_title"]
            text_col = next((c for c in fallbacks if c in df.columns), None)
            if text_col is None:
                continue

        df = df.rename(columns={text_col: "text"})

        # remove all empty text
        df = df.dropna(subset=["text"])
        df = df[df["text"].astype(str).str.strip() != ""]

        if not df.empty:
            dataframes.append(df[["datetime", "text"]])
            print(f"  Loaded {len(df)} entries from {filename}")

    if not dataframes:
        print(f"  No data found for {dev_path.name}")
        return pd.DataFrame(columns=["datetime", "text"])

    return pd.concat(dataframes, ignore_index=True).sort_values("datetime")


def compute_metric_series(series_or_df, smooth_window=4, mode="standard"):
    """
    Build the value/smoothed/std summary frame a signal is exported with.
      Params:
        series_or_df  (pd.Series or pd.DataFrame) : if mode is standard, fill,
                        or bar_ratio: a pd.Series indexed by month. If mode is
                        stacked: a DataFrame with one column per area layer,
                        returned unchanged (nothing to smooth).
        smooth_window (int)    : rolling-average window for the smoothed column
        mode          (string) : "standard" (default) / "fill" / "bar_ratio"
                        all build the same frame; "stacked" is a passthrough

      Returns:
        pd.DataFrame(['value', 'smoothed', 'std']), or the input DataFrame
        unchanged when mode == "stacked"
    """
    if mode == "stacked":
        return series_or_df

    series = series_or_df.copy().sort_index()
    series.index = pd.to_datetime(series.index)

    df = series.to_frame(name="value")
    df["smoothed"] = df["value"].rolling(window=smooth_window, min_periods=1).mean()
    df["std"]      = df["value"].rolling(window=smooth_window, min_periods=1).std()
    return df


def save_metric_data(dev_name, sheet_name, df):
    """
    Buffer metric data for export to Excel

      Params:
        dev_name(string)    : name of developers folder
        sheet_name(string)  : name of the sheet in the final Excel file
        df(pd.DataFrame)    : data to save
    """

    if dev_name not in excel_cache:
        excel_cache[dev_name] = {}

    out = df.reset_index()

    # Convert timezone-aware datetimes to naive
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            if hasattr(out[col].dt, "tz") and out[col].dt.tz is not None:
                out[col] = out[col].dt.tz_localize(None)

    excel_cache[dev_name][sheet_name] = out


def flush_metric_data(dev_folder):
    """
    Write all buffered metric DataFrames for a developer into a single Excel file
    Saved as dev_folder/dev_name_metrics.xlsx

      Params:
        dev_folder(Path) : developer folder
    """

    dev_name = dev_folder.name
    if dev_name not in excel_cache or not excel_cache[dev_name]:
        print(f"  No metric data to save for {dev_name}")
        return

    out_path = dev_folder / f"{dev_name}_metrics.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for sheet, df in excel_cache[dev_name].items():
            # Sheet names max 31 chars
            safe = sheet[:31]
            df.to_excel(writer, sheet_name=safe, index=False)
    print(f"  Metrics workbook saved: {out_path}")
    excel_cache.pop(dev_name, None)


## AROUSAL
Measures average emotional arousal via the NRC-VAD lexicon.

In [ ]:
# AROUSAL


#PATH TO VAD LEXICON
VAD_PATH = Path("")

def load_arousal_lexicon():
    """
    Load the NRC-VAD lexicon

    Returns:
     dictionary mapping from word to tuples containing arousal and valence scores string : (arousal, valence)

    """

    vad_df = pd.read_csv(VAD_PATH, sep="\t")
    vad_df.columns = vad_df.columns.str.lower()
    vad_df["term"] = vad_df["term"].str.lower()

    # Store both arousal and valence for weighted scoring
    vad_dict = {}
    for _, row in vad_df.iterrows():
        vad_dict[row["term"]] = (row["arousal"], row["valence"])

    print(f"Loaded {len(vad_dict)} VAD entries")
    return vad_dict


def compute_arousal(text):
    """
    Compute valence-weighted mean arousal for a single text
    Words with stronger valence (v closer to -1 or 1) get higher weight

      Params:
        text(string) : message text
        vad_dict (dict) : VAD dictionary(string: (float, float))

      Returns:
        float : valence-weighted mean arousal score
        NaN if no lexicon matches
    """

    # extract all words
    words = re.findall(r"\b[a-zA-Z]+\b", str(text).lower())
    hits  = [vad_dict[w] for w in words if w in vad_dict]
    if not hits:
        return np.nan

    arousals = np.array([h[0] for h in hits])
    weights  = np.abs(np.array([h[1] for h in hits]))

    if weights.sum() == 0:
        return float(np.mean(arousals))
    return float(np.average(arousals, weights=weights))


def process_arousal(dev_messages, vad_dict):
    """
    Compute monthly valence-weighted mean arousal

      Params:
        dev_messages(pd.DataFrame): dataframe with columns ["datetime","text"]
        vad_dict (dict) : VAD dictionary(string: (float, float))

      Returns:
        pd.Series of average monthly arousal indexed by year-month string
    """
    df = dev_messages.copy()
    df["arousal"] = df["text"].apply(compute_arousal)

    df = df.dropna(subset=["arousal"])
    df["year_month"] = df["datetime"].dt.to_period("M").astype(str)

    return df.groupby("year_month")["arousal"].mean()


def arousal(dev_folder, dev_messages, vad_dict):
    """
    Compute and buffer monthly arousal data for one developer

      Params:
        dev_folder(Path) : developer folder
        dev_messages(pd.DataFrame): Dataframe of all mesages
        vad_dict (dict) : VAD dictionary(string: (float, float))

      Returns:
        pd.DataFrame(['value', 'smoothed', 'std'])
        None if no valid data exists
    """

    dev_name = dev_folder.name
    print(f"\nProcessing Monthly Arousal for {dev_name}...")

    monthly = process_arousal(dev_messages, vad_dict)
    if monthly.empty:
        print(f"  No arousal data for {dev_name}")
        return None

    print(f"  {len(monthly)} months processed")

    result_df = compute_metric_series(monthly)

    if result_df is not None:
        save_metric_data(dev_name, "arousal", result_df)
    return result_df


vad_dict = load_arousal_lexicon()


## COMMITS
Monthly commit frequency with velocity (week-over-week change) overlay.

In [ ]:
# COMMITS

def process_commits(file_path):
    """
    Read the commits Excel file and compute monthly commit counts and velocity

      Params:
        file_path(Path) : path to a specific commits file

      Returns:
        tuple of pd.Series(counts, velocity) indexed by year-month string

    """
    df = pd.read_excel(file_path)

    if "date" not in df.columns:
        print(f"  Date column not found in {file_path}")
        return  pd.Series(dtype=float), pd.Series(dtype=float)

    df["date"] = pd.to_datetime(df["date"])
    df["year_month"] = df["date"].dt.to_period("M").astype(str)

    counts   = df.groupby("year_month").size()
    velocity = counts.diff()          # month-over-month change

    return counts, velocity


def commits(dev_folder):
    """
    Compute monthly commit frequency and velocity for one developer

    Params:
      dev_folder(Path) : developer folder

    """
    dev_name = dev_folder.name
    print(f"\nProcessing Monthly Commits for {dev_name}...")

    commit_file = dev_folder / f"{dev_name}_commits.xlsx"
    counts, velocity = process_commits(commit_file)

    print(f"  {len(counts)} months processed")

    result_df = compute_metric_series(counts)

    # combine both metrics into one DataFrame for excel export
    if result_df is not None:
        vel_df = velocity.to_frame(name="value").assign(
            smoothed=velocity.rolling(4, min_periods=1).mean()
        )

        combined = result_df.rename(columns={"value": "commit_count", "smoothed": "count_smoothed"})

        combined["velocity"] = vel_df["value"].values
        combined["velocity_smoothed"] = vel_df["smoothed"].values
        save_metric_data(dev_name, "commits", combined)


## PRONOUNS
Singular vs plural pronoun log-ratio

In [ ]:
# PRONOUNS

SINGULAR_PATTERN = r"\b(i|me|my|mine|myself)\b"
PLURAL_PATTERN   = r"\b(we|us|our|ours|ourselves)\b"


def process_pronouns(dev_messages):
    """
    Count singular and plural pronouns per month

      Params:
        dev_messages(pd.DataFrame) : DataFrame with columns ['datetime', 'text']

      Returns:
        pd.DataFrame with columns ['month', 'singular_count', 'plural_count', 'ratio', 'log_ratio', 'month_str']
    """
    df = dev_messages.copy()
    df["text"]  = df["text"].str.lower()

    # count via regex
    df["singular_count"] = df["text"].str.count(SINGULAR_PATTERN)
    df["plural_count"]   = df["text"].str.count(PLURAL_PATTERN)

    df["month"]          = df["datetime"].dt.to_period("M")

    monthly = df.groupby("month").agg(
        singular_count=("singular_count", "sum"),
        plural_count=("plural_count",   "sum"),
    ).reset_index()

    total = monthly["singular_count"] + monthly["plural_count"]
    monthly["ratio"] = monthly["singular_count"] / total.replace(0, np.nan)

    # log ratio + laplace smoothing to avoid 0 divison
    monthly["log_ratio"] = np.log(
        (monthly["singular_count"] + 1) / (monthly["plural_count"] + 1)
    )
    monthly["month_str"] = monthly["month"].astype(str)

    print(f"  Processed {len(monthly)} months of pronoun data")
    return monthly


def pronouns(dev_folder, dev_messages):
    """
    Compute monthly pronoun log-ratio for one developer

      Params:
        dev_folder(Path) : developer folder
        dev_messages(pd.DataFrame) : DataFrame with columns ['datetime', 'text']
    """
    dev_name = dev_folder.name
    print(f"\nProcessing Monthly Pronoun Use for {dev_name}...")

    monthly = process_pronouns(dev_messages)
    save_metric_data(dev_name, "pronouns", monthly)


## LEXICAL DIVERSITY
MATTR — moving-average type-token ratio.

In [ ]:
# LEXICAL DIVERSITY

def tokenize(text):
    """
    Extract alphabetic words (lowercase) from text
    """
    return re.findall(r"\b[a-z]+\b", str(text).lower())


def compute_ttr(tokens):
  """
  TTR = unique tokens / total tokens
  """
  return len(set(tokens)) / len(tokens) if tokens else np.nan


def compute_mattr(tokens, window=50):
    """
    Moving-Average Type-Token Ratio
    For texts shorter than window, returns standard TTR
    """

    if len(tokens) < window:
        return compute_ttr(tokens)

    scores = []

    for i in range(len(tokens) - window + 1):
        window_tokens = tokens[i : i + window]
        unique_tokens = set(window_tokens)
        score = len(unique_tokens) / window
        scores.append(score)

    return float(np.mean(scores))


def compute_hapax_rate(tokens):
    """
    Fraction of unique word types that appear exactly once
    """
    from collections import Counter
    freq  = Counter(tokens)
    if not freq:
        return np.nan
    hapax = sum(1 for v in freq.values() if v == 1)
    return hapax / len(freq)


def analyze_lexical_diversity(dev_messages):
    """
    Compute monthly TTR, MATTR, and hapax rate

      Params:
        dev_messages(pd.DataFrame) : DataFrame with columns ['datetime', 'text']

      Returns:
        pd.DataFrame ['month', 'ttr', 'mattr', 'hapax_rate', 'token_count', 'mattr_smooth', 'hapax_rate_smooth', 'month_str']

    """
    df = dev_messages.copy()
    df["tokens"] = df["text"].apply(tokenize)
    df["month"]  = df["datetime"].dt.to_period("M")

    rows = []
    for month, group in df.groupby("month"):
        all_tokens = []
        for ts in group["tokens"]:
            for t in ts:
                all_tokens.append(t)

        rows.append({
            "month":       month,
            "ttr":         compute_ttr(all_tokens),
            "mattr":       compute_mattr(all_tokens, window=50),
            "hapax_rate":  compute_hapax_rate(all_tokens),
            "token_count": len(all_tokens),
        })

    monthly = pd.DataFrame(rows).sort_values("month")

    # 3-month rolling window smoothing
    monthly["mattr_smooth"]      = monthly["mattr"].rolling(3, min_periods=1).mean()
    monthly["hapax_rate_smooth"] = monthly["hapax_rate"].rolling(3, min_periods=1).mean()
    monthly["month_str"]         = monthly["month"].astype(str)
    return monthly


def lexicalDiversity(dev_folder, dev_messages):
    """
    Compute monthly lexical diversity for one developer

      Params:
        dev_folder(Path) : developer folder
        dev_messages(pd.DataFrame) : DataFrame with columns ['datetime', 'text']
    """
    dev_name  = dev_folder.name
    print(f"\nProcessing Lexical Diversity for {dev_name}...")

    monthly  = analyze_lexical_diversity(dev_messages)

    print(f"  {len(monthly)} months processed")
    save_metric_data(dev_name, "lexical_diversity", monthly)


## LEXICAL FEATURES
Semantic similarity scoring for exhaustion / disengagement seeds.

In [ ]:
# LEXICAL FEATURES (LF)

from sentence_transformers import SentenceTransformer

#  exhaustion (physical, cognitive, emotional fatigue)
EXHAUSTION_SEEDS = [
    "completely drained", "running on empty", "burned out", "mentally exhausted",
    "can't focus", "losing focus", "can't concentrate", "brain fog",
    "too tired to", "no energy left", "pushing through",
    "overloaded", "swamped", "buried in", "drowning in work",
    "working nights", "working weekends", "no time to breathe",
    "stretched thin", "taking on too much",
    "don't have the bandwidth", "out of bandwidth", "at my limit",
    "running low", "wearing me down", "wearing me out",
    "this is exhausting", "so much pressure", "under a lot of pressure",
    "good enough", "ship it", "just make it work", "doesn't need to be perfect",
    "too tired to care", "whatever works",
]

#  disengagement (withdrawal, cynicism, reduced effort)
DISENGAGEMENT_SEEDS = [
    "closing this", "closing without merging", "abandoning this",
    "won't be continuing", "not going to pursue", "dropping this",
    "no longer interested", "moving on from this", "stepping back",
    "not my problem", "someone else can", "not my responsibility",
    "not worth my time", "don't see the point", "pointless exercise",
    "going through the motions", "just checking the box",
    "lost motivation", "no longer motivated", "don't care anymore",
    "stopped caring", "can't bring myself to", "hard to stay motivated",
    "what's the point", "does it even matter", "nobody uses this anyway",
    "this codebase is hopeless", "waste of time", "nothing ever changes",
    "same issues again", "tired of repeating myself",
    "not listening", "ignored again", "falling on deaf ears",
    "leaving it to you", "you decide", "I'll let the team handle",
    "not going to review", "skipping the review",
]

lf_model = SentenceTransformer("all-MiniLM-L6-v2")

# precompute all seed embeddings
seed_embeddings = {
    "exhaustion":    lf_model.encode(EXHAUSTION_SEEDS,    convert_to_numpy=True),
    "disengagement": lf_model.encode(DISENGAGEMENT_SEEDS, convert_to_numpy=True),
}


def cosine_similarity(a, b):
  """
  Compute cosine similarity between two vectors
  """
  return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


def avg_max_seed_similarity(sentenceEmbedding, seedEmbs):
    """
    return the mean similarity of the top-3 closest seeds to a sentence embedding

    """
    sims_list = []

    for s in seedEmbs:
        sim = cosine_similarity(sentenceEmbedding, s)
        sims_list.append(sim)

    sims = sorted(sims_list, reverse=True)

    return float(np.mean(sims[:3]))


def extract_features(text):
    """
    Score one message against both exhaustion and disengagement seed sets

      Params:
        text(string): message text

      Returns:
        dict, Keys: "exhaustion", "disengagement", Values: average similarity score
    """
    text = clean_text(text)

    sentences = []

    for s in text.split("."):
        stripped = s.strip()
        if stripped:
            sentences.append(stripped)

    # if nothing valid was found, return original text as a single sentence
    if not sentences:
        sentences = [text]

    sent_embs = lf_model.encode(sentences, convert_to_numpy=True)

    result = {}
    for category, seeds in seed_embeddings.items():
        similarities = []

        for emb in sent_embs:
            sim = avg_max_seed_similarity(emb, seeds)
            similarities.append(sim)

        mean_sim = np.mean(similarities)
        result[category] = float(mean_sim)

    return result


def analyze_exhaustion(df):
  """
  Add exhaustion and disengagement scores to each message

  """
  df = df.copy()
  df["cleaned"] = df["text"].apply(clean_text)
  features  = df["cleaned"].apply(extract_features)
  return pd.concat([df, pd.DataFrame(list(features))], axis=1)


def monthly_trend(df):
  """
  Aggregate daily scores to monthly averages
  """
  df = df.copy()
  df["month"] = df["datetime"].dt.to_period("M").dt.to_timestamp()
  return df.groupby("month").agg(
      exhaustion=("exhaustion",    "mean"),
      disengagement=("disengagement", "mean"),
  ).reset_index()


def LF(dev_folder, dev_messages):
    """
    Compute monthly lexical feature trends for one developer

    Params:
        dev_folder(Path) : developer folder
        dev_messages(pd.DataFrame) : DataFrame with columns ['datetime', 'text']
    """
    dev_name = dev_folder.name
    print(f"\nProcessing Lexical Features for {dev_name}...")

    df = analyze_exhaustion(dev_messages)
    monthly = monthly_trend(df)

    print(f"  {len(monthly)} months processed")
    save_metric_data(dev_name, "lexical_features", monthly)


## OFF-HOURS ACTIVITY
Inferred work-window, outside-hours rate, and temporal entropy.

In [ ]:
# OFF-HOURS ACTIVITY

def load_timestamps(dev_folder):
    """
    Load all activity timestamps for a developer

      Params:
        dev_name: Path to developer folder

      Returns:
        pd.DataFrame with columns ['datetime']

    """
    dev_name = dev_folder.name
    files = [
        (f"{dev_name}_commits.xlsx",        "date"),
        (f"{dev_name}_issue_comments.xlsx",  "created_at"),
        (f"{dev_name}_prs.xlsx",             "created_at"),
        (f"{dev_name}_review_comments.xlsx", "created_at"),
        (f"{dev_name}_reviews.xlsx",         "submitted_at"),
    ]

    dataframes = []
    for filename, col in files:
        path = dev_folder / filename
        if not path.exists():
            continue

        df = pd.read_excel(path)
          # Find the first existing date column
        date_col = None
        possible_cols = ['date', 'created_at', 'timestamp', 'datetime', 'committed_date']
        for col in possible_cols:
            if col in df.columns:
                date_col = col
                break
        if date_col is None:
            print(f"    No date column found in {filename}, skipping.")
            continue

        df["datetime"] = pd.to_datetime(df[date_col], errors="coerce")
        df = df.dropna(subset=["datetime"])
        if not df.empty:
          dataframes.append(df[["datetime"]])

    if not dataframes:
        return pd.DataFrame(columns=["datetime"])
    return pd.concat(dataframes).sort_values("datetime")


def get_window(df, window_size=10):
    """
    Find the most active N-hour window in the 24-hour day by sliding sum

      Params:
        df (pd.DataFrame): dataframe of developer activity timestamps
        window_size(int): size of the window

      Returns:
        tuple of start and end hour indices

    """
    hour_counts = (
        df["datetime"].dt.hour.value_counts()
        .reindex(range(24), fill_value=0)
        .sort_index()
        .values
    )

    # duplicate to handle wrap around cases
    extended = np.concatenate([hour_counts, hour_counts])

    # Find window with most activity
    window_sums = []
    for i in range(24):
        window_slice = extended[i : i + window_size]
        window_sum = window_slice.sum()
        window_sums.append(window_sum)

    start = int(np.argmax(window_sums))
    return start, (start + window_size) % 24


def monthly_outside_series(df, window_size=10, min_events=10):
    """
    Compute the monthly outside-work rate as a Series, interpolating missing months

      Params:
        df(pd.DataFrame): column ['datetime'] with all dev timestamps
        window_size(int): size of time window
        min_events(int): minimum number of events that occured in a given month to be considered

      Returns:
        pd.Series with monthly outside-work rate
    """
    df = df.copy()
    df["hour"] = df["datetime"].dt.hour
    start, end = get_window(df, window_size)

    # Determine if a timestamp falls outside the work window
    if start < end:
      outside_mask = (df["hour"] < start) | (df["hour"] >= end)
    else:
      outside_mask = (df["hour"] < start) & (df["hour"] >= end)

    df["outside"] = outside_mask.astype(int)
    df = df.set_index("datetime")

    # Monthly outside rate
    monthly_total   = df.resample("M").size()
    monthly_outside = df["outside"].resample("M").sum()
    counts = monthly_outside / monthly_total

    # Exclude months with few events
    counts[monthly_total < min_events] = np.nan

    return counts.interpolate().sort_index()


def monthly_entropy_series(df):
    """
    Compute monthly Shannon entropy of hourly activity

      Params:
        df(pd.DataFrame): column ['datetime'] with all dev timestamps

      returns:
        pd.Series with monthly Shannon entropy
    """
    df = df.copy().set_index("datetime")
    df["hour"] = df.index.hour

    months = df.resample("M")
    results = {}
    for period, group in months:
        if len(group) < 5:
            results[period] = np.nan
            continue
        freq = group["hour"].value_counts(normalize=True)
        freq = freq[freq > 0]
        results[period] = -(freq * np.log2(freq)).sum()

    return pd.Series(results)


def off_hours(dev_folder):
    """
    Compute monthly off-hours rate and temporal entropy for one developer

    Params:
      dev_folder: Path to developer folder
    """

    dev_name = dev_folder.name
    print(f"\nProcessing Off-Hours Activity for {dev_name}...")

    df = load_timestamps(dev_folder)

    # Outside work rate
    outside_series = monthly_outside_series(df)
    result_df = compute_metric_series(outside_series)

    entropy_series = monthly_entropy_series(df)
    entropy_df = compute_metric_series(entropy_series)

    if result_df is not None:
        export_df = result_df.rename(columns={
            "value": "outside_rate", "smoothed": "outside_rate_smoothed"
        })[["outside_rate", "outside_rate_smoothed"]]

        if entropy_df is not None:
            entropy_export = entropy_df.rename(columns={
                "value": "temporal_entropy", "smoothed": "temporal_entropy_smoothed"
            })[["temporal_entropy", "temporal_entropy_smoothed"]]
            export_df = export_df.join(entropy_export, how="outer")

        save_metric_data(dev_name, "off_hours", export_df)


## PR THROUGHPUT
Rejected PRs, reviews submitted, and average merge time.

In [ ]:
# PR THROUGHPUT

def processPRs(file_path):
    """
    Read PR data and compute monthly rejected PR counts and average merge time

      Params:
        file_path(Path): path to PRs file

      Returns:
        tuple of pd.Series (rejected, merge time)
    """
    df = pd.read_excel(file_path)

    # remove timezone info
    if ("created_at" not in df.columns) or ("merged_at" not in df.columns):
        print(f"  No valid date columns in {file_path.name}")
        return pd.Series(dtype=int), pd.Series(dtype=float)

    df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
    df["merged_at"]  = pd.to_datetime(df["merged_at"], errors="coerce").dt.tz_localize(None)
    df["year_month"] = df["created_at"].dt.to_period("M").astype(str)

    # Rejected prs (closed but not merged)
    rejected = df[(df["state"] == "closed") & df["merged_at"].isna()]
    rejected_counts = rejected.groupby("year_month").size()

    # Merged Prs
    merged = df[df["merged_at"].notna()].copy()

    # avg merge time
    merged["merge_time"] = (
        (merged["merged_at"] - merged["created_at"]).dt.total_seconds() / 86400
    )
    avg_merge_time = merged.groupby("year_month")["merge_time"].mean()

    return rejected_counts, avg_merge_time


def processReviews(file_path):
    """
    Count reviews per month

    Params:
      file_path(Path): path to reviews file

    Returns:
      pd.Series with monthly review counts
    """
    df = pd.read_excel(file_path)
    if df.empty:
        return pd.Series(dtype=int)

    # find the date column
    date_col = next(
        (c for c in ["submitted_at", "created_at", "date"] if c in df.columns), None
    )
    if date_col is None:
        print(f"  No valid date column in {file_path.name}")
        return pd.Series(dtype=int)

    df["datetime"]   = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=["datetime"])
    df["year_month"] = df["datetime"].dt.to_period("M").astype(str)

    return df.groupby("year_month").size()


def normalize_series(series):
    """Reindex to a continuous monthly range. Fills empty months with 0"""
    if series.empty:
        return series

    series.index = pd.to_datetime(series.index)
    full_range   = pd.date_range(series.index.min(), series.index.max(), freq="MS")

    return series.reindex(full_range, fill_value=0)


def PRThroughput(dev_folder):
    """
    Compute monthly PR throughput metrics for one developer

      Params:
        dev_folder(Path): path to developer folder
    """
    dev_name = dev_folder.name
    print(f"\nProcessing PR Throughput for {dev_name}...")

    pr_path     = dev_folder / f"{dev_name}_prs.xlsx"
    review_path = dev_folder / f"{dev_name}_reviews.xlsx"

    rejected, merge_time = processPRs(pr_path)
    reviews              = processReviews(review_path)

    # Ensure all series cover the same range
    rejected   = normalize_series(rejected)
    merge_time = normalize_series(merge_time)
    reviews    = normalize_series(reviews)

    r1 = compute_metric_series(rejected)
    r2 = compute_metric_series(reviews)
    r3 = compute_metric_series(merge_time)

    # Combine into one DataFrame for export
    export = None
    if r1 is not None:
        export = r1[["value", "smoothed"]].rename(
            columns={"value": "rejected_prs", "smoothed": "rejected_smooth"}
        )
    if r2 is not None:
        r2_df = r2[["value", "smoothed"]].rename(
            columns={"value": "reviews", "smoothed": "reviews_smooth"}
        )
        export = r2_df if export is None else export.join(r2_df, how="outer")
    if r3 is not None:
        r3_df = r3[["value", "smoothed"]].rename(
            columns={"value": "avg_merge_days", "smoothed": "merge_days_smooth"}
        )
        export = r3_df if export is None else export.join(r3_df, how="outer")

    if export is not None:
        save_metric_data(dev_name, "pr_throughput", export)


## TASK ABANDONMENT
Abandoned PR counts, abandonment rate, and stale open PRs.

In [ ]:
# TASK ABANDONMENT

def process_task_abandonment(file_path):
    """
    Process PR data to compute monthly abandoned PR counts, abandonment rate,
    and stale open PR counts

      Params:
        file_path(Path): path to PRs file

      Returns:
        tuple of pd.Series (abandoned, abandonment rate, stale)
    """

    df = pd.read_excel(file_path)
    if ("created_at" not in df.columns) or ("merged_at" not in df.columns):
      print(f"  No valid date columns in {file_path.name}")
      return pd.Series(dtype=float), pd.Series(dtype=float), pd.Series(dtype=float)

    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce").dt.tz_localize(None)
    df["merged_at"]  = pd.to_datetime(df["merged_at"],  errors="coerce").dt.tz_localize(None)

    if "closed_at" in df.columns:
        df["closed_at"] = pd.to_datetime(df["closed_at"], errors="coerce")
        # Strip timezone only if the column is datetime (not all-NaT object)
        if pd.api.types.is_datetime64_any_dtype(df["closed_at"]):
            df["closed_at"] = df["closed_at"].dt.tz_localize(None)

    df = df.dropna(subset=["created_at"])
    df = df.sort_values("created_at")
    df["year_month"] = df["created_at"].dt.to_period("M").astype(str)  # always a string

    total = df.groupby("year_month").size()
    abandoned = df[(df["state"] == "closed") & df["merged_at"].isna()]
    abandoned_counts = abandoned.groupby("year_month").size()
    abandonment_rate = (abandoned_counts / total).fillna(0)

    # Stale, still open but created > 60 days ago
    reference_date = df["created_at"].max()
    stale = df[
        (df["state"] == "open") &
        ((reference_date - df["created_at"]).dt.days > 60)
    ]
    stale_counts = stale.groupby("year_month").size()

    return abandoned_counts, abandonment_rate, stale_counts


def task_abandonment(dev_folder):
    """
    Compute monthly PR abandonment metrics for one developer

      Params:
        dev_folder(Path): path to developer folder
    """
    dev_name = dev_folder.name
    print(f"\nProcessing Task Abandonment for {dev_name}...")

    pr_file = dev_folder / f"{dev_name}_prs.xlsx"
    abandoned_counts, abandonment_rate, stale_counts = process_task_abandonment(pr_file)

    r1 = compute_metric_series(abandoned_counts)
    r2 = compute_metric_series(abandonment_rate)
    r3 = compute_metric_series(stale_counts)

    if r1 is not None:
        export = r1[["value", "smoothed"]].rename(
            columns={"value": "abandoned_count", "smoothed": "abandoned_smooth"}
        )
        if r2 is not None:
            rate_df = r2[["value", "smoothed"]].rename(
                columns={"value": "abandonment_rate", "smoothed": "abandonment_rate_smooth"}
            )
            export = export.join(rate_df, how="left")
        if r3 is not None:
            stale_df = r3[["value", "smoothed"]].rename(
                columns={"value": "stale_prs", "smoothed": "stale_prs_smooth"}
            )
            export = export.join(stale_df, how="left")
        save_metric_data(dev_name, "task_abandonment", export)


## SENTIMENT ANALYSIS
RoBERTa classifier — monthly sentiment score.

In [ ]:
# SENTIMENT ANALYSIS

import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# BEST FOLD DIR
CHECKPOINT_DIR = ""
MAX_LENGTH     = 256
BATCH_SIZE     = 64
LABEL2ID = {"negative": 0, "positive": 1, "neutral": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

print(f"Loading RoBERTa checkpoint from: {CHECKPOINT_DIR}")
tokenizer = RobertaTokenizer.from_pretrained(CHECKPOINT_DIR)
model     = RobertaForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)
model.eval()
print("Model loaded.")


def predict_batch(texts):
    """
    Run fine tuned RoBERTa on a batch of texts

      Params:
        texts(list): a list of developer texts

      Returns:
        list of dicts with sentiment label, confidence, and scores
    """
    cleaned = [clean_text(t) for t in texts]
    encodings = tokenizer(
        cleaned, max_length=MAX_LENGTH, padding="max_length",
        truncation=True, return_tensors="pt",
    )

    with torch.no_grad():
        outputs = model(
            input_ids=encodings["input_ids"].to(DEVICE),
            attention_mask=encodings["attention_mask"].to(DEVICE),
        )
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=-1)

    return [
        {
            "sentiment":  ID2LABEL[pred],
            "confidence": float(prob[pred]),
            "neg_score":  float(prob[0]),
            "pos_score":  float(prob[1]),
            "neu_score":  float(prob[2]),
        }
        for pred, prob in zip(preds, probs)
    ]


def analyze_sentiment(dev_folder):
    """
    Run RoBERTa over all messages and aggregate to monthly stats

      Params:
        dev_folder(Path): path to developer folder

      Returns:
        pd.DataFrame with monthly stats or None
    """

    dev_name = dev_folder.name
    print(f"\nAnalyzing sentiment for {dev_name}...")

    df = load_messages(dev_folder)
    if df.empty:
        print(f"  No data for {dev_name}")
        return None

    texts = df["text"].astype(str).tolist()

    all_preds = []
    for i in range(0, len(texts), BATCH_SIZE):
        all_preds.extend(predict_batch(texts[i: i + BATCH_SIZE]))
        print(f"    {min(i + BATCH_SIZE, len(texts))} / {len(texts)} done")

    df = pd.concat([df.reset_index(drop=True), pd.DataFrame(all_preds)], axis=1)
    df["month"] = df["datetime"].dt.to_period("M")

    monthly = df.groupby("month").agg(
        total_messages=("sentiment", "count"),
        neg_count     =("sentiment", lambda x: (x == "negative").sum()),
        pos_count     =("sentiment", lambda x: (x == "positive").sum()),
        neu_count     =("sentiment", lambda x: (x == "neutral").sum()),
        avg_neg_score =("neg_score", "mean"),
        avg_pos_score =("pos_score", "mean"),
        avg_neu_score =("neu_score", "mean"),
    ).reset_index()

    monthly["neg_ratio"] = monthly["neg_count"] / monthly["total_messages"]
    monthly["pos_ratio"] = monthly["pos_count"] / monthly["total_messages"]
    monthly["neu_ratio"] = monthly["neu_count"] / monthly["total_messages"]
    monthly["sentiment_score"] = monthly["avg_pos_score"] - monthly["avg_neg_score"]
    monthly["log_pos_neg_ratio"] = np.log(
        (monthly["pos_count"] + 1) / (monthly["neg_count"] + 1)
    )

    # 3 month rolling avg smoothing
    for col in ["sentiment_score", "log_pos_neg_ratio", "neg_ratio", "pos_ratio"]:
        monthly[f"{col}_smooth"] = monthly[col].rolling(3, min_periods=1).mean()
    monthly["month_str"] = monthly["month"].astype(str)

    print(f"  {len(monthly)} months processed")
    return monthly


def compute_sentiment(dev_folder):
    """
    Compute monthly sentiment stats for one developer and buffer for export.

    Params:
      dev_folder(Path): path to developer folder

    Returns:
      dict summary of the developer's overall sentiment, or None
    """
    dev_name = dev_folder.name
    monthly  = analyze_sentiment(dev_folder)
    if monthly is None or monthly.empty:
        return None

    save_metric_data(dev_name, "sentiment", monthly)
    return {
        "developer":           dev_name,
        "avg_sentiment_score": monthly["sentiment_score_smooth"].mean(),
        "trend_direction": (
            "improving"
            if monthly["sentiment_score_smooth"].iloc[-1]
               > monthly["sentiment_score_smooth"].iloc[0]
            else "declining"
        ),
        "months_analyzed":  len(monthly),
        "total_messages":   int(monthly["total_messages"].sum()),
        "total_negative":   int(monthly["neg_count"].sum()),
        "total_positive":   int(monthly["pos_count"].sum()),
        "total_neutral":    int(monthly["neu_count"].sum()),
        "overall_neg_ratio": monthly["neg_count"].sum() / monthly["total_messages"].sum(),
        "overall_pos_ratio": monthly["pos_count"].sum() / monthly["total_messages"].sum(),
    }


## BURNOUT CALCULATION
Turns the signal sheets above into monthly
Exhaustion / Disengagement / Combined risk scores for one developer, finds
every sustained high-risk onset, and measures the departure / activity-decline
/ role-withdrawal outcomes that followed each one.



In [ ]:
# BURNOUT CALCULATION
# =============================================================================
# (sheet_name, column_name, direction, dimension, weight_profile_key)
# =============================================================================

WEIGHT_PROFILES = {
    "default": {
        "off_hours/outside_rate":            1.387,
        "arousal/value":                     0.651,
        "pr_throughput/avg_merge_days":      0.594,
        "lexical_features/exhaustion":       1.100,
        "sentiment/sentiment_score_E":       0.900,
        "commits/commit_count":              1.536,
        "pr_throughput/reviews":             0.217,
        "pr_throughput/rejected_prs":        0.192,
        "task_abandonment/abandonment_rate": 0.523,
        "pronouns/log_ratio":                0.9,
        "lexical_features/disengagement":    1.1,
        "lexical_diversity/mattr":           0.9,
        "lexical_diversity/hapax_rate":      0.05,
        "sentiment/sentiment_score_D":       0.9,
    },
    "exhaustion_heavy": {
        "off_hours/outside_rate":            3.033,
        "arousal/value":                     1.273,
        "pr_throughput/avg_merge_days":      0.330,
        "lexical_features/exhaustion":       2.194,
        "sentiment/sentiment_score_E":       1.796,
        "commits/commit_count":              1.658,
        "pr_throughput/reviews":             0.050,
        "pr_throughput/rejected_prs":        0.050,
        "task_abandonment/abandonment_rate": 0.138,
        "pronouns/log_ratio":                0.9,
        "lexical_features/disengagement":    1.1,
        "lexical_diversity/mattr":           0.9,
        "lexical_diversity/hapax_rate":      0.05,
        "sentiment/sentiment_score_D":       0.9,
    },
    "disengagement_heavy": {
       "off_hours/outside_rate":             5.000,
        "arousal/value":                     0.050,
        "pr_throughput/avg_merge_days":      0.050,
        "lexical_features/exhaustion":       1.000,
        "sentiment/sentiment_score_E":       1.000,
        "commits/commit_count":              5.000,
        "pr_throughput/reviews":             0.050,
        "pr_throughput/rejected_prs":        0.050,
        "task_abandonment/abandonment_rate": 0.050,
        "pronouns/log_ratio":                1.000,
        "lexical_features/disengagement":    1.000,
        "lexical_diversity/mattr":           1.000,
        "lexical_diversity/hapax_rate":      1.000,
        "sentiment/sentiment_score_D":       1.000,
    },
    "equal": {
        "off_hours/outside_rate":            1.251,
        "arousal/value":                     0.525,
        "pr_throughput/avg_merge_days":      0.062,
        "lexical_features/exhaustion":       1.0,
        "sentiment/sentiment_score_E":       1.0,
        "commits/commit_count":              1.470,
        "pr_throughput/reviews":             0.05,
        "pr_throughput/rejected_prs":        0.05,
        "task_abandonment/abandonment_rate": 0.05,
        "pronouns/log_ratio":                1.0,
        "lexical_features/disengagement":    1.0,
        "lexical_diversity/mattr":           1.0,
        "lexical_diversity/hapax_rate":      0.05,
        "sentiment/sentiment_score_D":       1.0,
    },
}

SIGNAL_DEFS = [
    ("off_hours",        "outside_rate",     +1, "E", "off_hours/outside_rate"),
    ("arousal",          "value",            -1, "E", "arousal/value"),
    ("pr_throughput",    "avg_merge_days",   +1, "E", "pr_throughput/avg_merge_days"),
    ("lexical_features", "exhaustion",       +1, "E", "lexical_features/exhaustion"),
    ("sentiment",        "sentiment_score",   0, "E", "sentiment/sentiment_score_E"),
    ("commits",          "commit_count",     -1, "D", "commits/commit_count"),
    ("pr_throughput",    "reviews",          -1, "D", "pr_throughput/reviews"),
    ("pr_throughput",    "rejected_prs",     +1, "D", "pr_throughput/rejected_prs"),
    ("pronouns",         "log_ratio",        +1, "D", "pronouns/log_ratio"),
    ("task_abandonment", "abandonment_rate", +1, "D", "task_abandonment/abandonment_rate"),
    ("lexical_features", "disengagement",    +1, "D", "lexical_features/disengagement"),
    ("lexical_diversity","mattr",            -1, "E", "lexical_diversity/mattr"),
    ("lexical_diversity","hapax_rate",       -1, "E", "lexical_diversity/hapax_rate"),
    ("sentiment",        "sentiment_score",  -1, "D", "sentiment/sentiment_score_D"),
]

# Monthly burnout score threshold to flag a developer as "at risk" that month.
RISK_THRESHOLD = 0.411

# After a burnout onset is detected, how many months must pass before a new
# onset can be flagged again. Without this, a single sustained high-risk
# stretch would otherwise re-trigger every month it stays above threshold.
ONSET_COOLDOWN_MONTHS = 12

# A developer is considered to have "departed" if they have this many or more
# consecutive months of zero activity after a high-risk month.
DEPARTURE_SILENCE_MONTHS = 3

# Minimum fractional drop in activity to count as a significant decline.
DECLINE_THRESHOLD = 0.40

# How many months before and after the first high-risk month to use as the
# comparison windows for activity change.
LOOKBACK_MONTHS  = 6   # baseline: 6 months before risk onset
LOOKAHEAD_MONTHS = 6   # outcome:  6 months after risk onset

# Minimum months of activity required in the baseline window to be included.
MIN_BASELINE_MONTHS = 3

# Minimum average monthly activity in the baseline period for a developer to
# be scored on outcomes at all — filters out drive-by contributors who were
# never meaningfully active.
MIN_BASELINE_ACTIVITY = 2.0   # events per month on average

# Weight profile to score with. See WEIGHT_PROFILES above for alternatives.
EVAL_PROFILE = "equal"

# Raw activity files per developer — (filename_suffix, date_column).
ACTIVITY_FILES = [
    ("_commits.xlsx",         "date"),
    ("_prs.xlsx",             "created_at"),
    ("_reviews.xlsx",         "submitted_at"),
    ("_issue_comments.xlsx",  "created_at"),
    ("_review_comments.xlsx", "created_at"),
]




def expanding_percentile(series):
    """
    For each point, compute its percentile rank among all prior points
    (expanding window).

      Params:
        series(pd.Series): monthly values of a single metric

      Returns:
        pd.Series with percentile ranks
    """
    result = []
    for i in range(len(series)):
        window = series.iloc[:i + 1].dropna()
        if len(window) < 3:
            result.append(np.nan)
        else:
            result.append(float((window < series.iloc[i]).sum() / len(window)))
    return pd.Series(result, index=series.index)


def slope_signal(series, window=4):
    """
    Rolling linear regression slope over `window` months

      Params:
        series(pd.Series): monthly values
        window(int): number of months to regress over

      Returns:
        pd.Series with the slope for each month
    """
    slopes = []
    for i in range(len(series)):
        if i < window - 1:
            slopes.append(np.nan)
        else:
            y = series.iloc[i - window + 1:i + 1].values.astype(float)
            if np.isnan(y).any():
                slopes.append(np.nan)
            else:
                slopes.append(float(np.polyfit(np.arange(window), y, 1)[0]))
    return pd.Series(slopes, index=series.index)


def sigmoid(x, center=0.65, steepness=8):
    return 1 / (1 + np.exp(-steepness * (x - center)))


def score_signal(series, direction):
    """
    Convert a raw metric series into a [0,1] burnout score.

      Params:
        series(pd.Series): monthly values of a single metric
        direction(int): whether increasing (+1) or decreasing (-1) values
                        indicate burnout

      Returns:
        pd.Series with the burnout contribution score per month
    """
    s = series * direction
    level_pct = expanding_percentile(s)
    slope_pct = expanding_percentile(
        slope_signal(s).bfill().fillna(0)
    )
    combined = (level_pct * 0.5 + slope_pct * 0.5).fillna(0)
    return combined.ewm(span=4, min_periods=2).mean().apply(sigmoid)


def neutralisation_score(series):
    """Special case for signed signals where either
    extreme, not just one direction, can indicate burnout."""
    return score_signal(-series.abs(), direction=+1)


def load_sheet(metrics_path, sheet):
    """
    Load one sheet from the metrics workbook, indexed by month where possible.

      Params:
        metrics_path(Path): path to the developer's metrics workbook
        sheet(string): sheet name to load

      Returns:
        pd.DataFrame, or None if the sheet doesn't exist
    """
    try:
        xf = pd.ExcelFile(metrics_path)
        candidates = [s for s in xf.sheet_names if s[:31] == sheet[:31]]
        if not candidates:
            return None
        df = pd.read_excel(metrics_path, sheet_name=candidates[0])
        for col in ["index", "month", "year_month", "month_str", "datetime"]:
            if col in df.columns:
                df["_period"] = pd.to_datetime(df[col], errors="coerce")
                df = df.dropna(subset=["_period"]).set_index("_period").sort_index()
                break
        return df
    except Exception:
        return None


def score_developer_full(devFolder, profile):
    """
    Read every signal sheet from the developer's metrics workbook and
    combine them into a single monthly Combined Risk series.

      Params:
        devFolder(Path): developer folder (must already have
                          {dev_name}_metrics.xlsx from the signal cells above)
        profile(dict): one of WEIGHT_PROFILES' entries

      Returns:
        pd.Series of monthly combined risk score in [0, 1], or None if no
        signal could be scored
    """
    metrics_path = devFolder / f"{devFolder.name}_metrics.xlsx"
    if not metrics_path.exists():
        return None

    e_scores, e_weights = [], []
    d_scores, d_weights = [], []

    for sheet, col, direction, dim, profile_key in SIGNAL_DEFS:
        df = load_sheet(metrics_path, sheet)
        if df is None or col not in df.columns:
            continue
        series = (
            df[col].dropna().astype(float)
            .resample("MS").mean()
            .interpolate(limit=2)
        )
        if len(series) < 3:
            continue
        score = (
            neutralisation_score(series)
            if direction == 0
            else score_signal(series, direction)
        )
        w = profile.get(profile_key, 1.0)
        if dim == "E":
            e_scores.append(score)
            e_weights.append(w)
        else:
            d_scores.append(score)
            d_weights.append(w)

    def _combine(scores, weights):
        if not scores:
            return pd.Series(dtype=float)
        df_all = pd.concat(scores, axis=1).resample("MS").mean()
        w = np.array(weights[:df_all.shape[1]])
        w = w / w.sum()
        return df_all.mul(w).sum(axis=1, min_count=1)

    E = _combine(e_scores, e_weights)
    D = _combine(d_scores, d_weights)
    if E.empty and D.empty:
        return None

    common = E.index.union(D.index)
    E = E.reindex(common).interpolate(limit=2)
    D = D.reindex(common).interpolate(limit=2)
    return ((E + D) / 2).dropna()


def strip_tz(series):
    """Strip timezone from a datetime Series, converting to UTC first if tz-aware."""
    if hasattr(series.dt, "tz") and series.dt.tz is not None:
        series = series.dt.tz_convert("UTC").dt.tz_localize(None)
    return series


def to_monthly_series(timestamps):
    """
    Convert a Series of datetimes to a monthly count Series with a clean
    tz-naive DatetimeIndex (MS freq).
    """
    ts = strip_tz(timestamps).dropna()
    if ts.empty:
        return pd.Series(dtype=float)
    counts = (
        ts.sort_values()
        .rename("_dt")
        .to_frame()
        .set_index("_dt")
        .assign(_n=1)["_n"]
        .resample("MS")
        .sum()
    )
    counts.index = pd.to_datetime(counts.index).tz_localize(None)
    return counts


def normalize_index(s):
    """Force any Series index to a tz-naive DatetimeIndex.

    Handles the empty-series case explicitly: an empty Series (e.g. from a
    developer with no _commits.xlsx / _reviews.xlsx file) otherwise keeps
    its default RangeIndex, which blows up with a TypeError the first time
    it's compared against a Timestamp downstream.
    """
    s = s.copy()
    if s.empty:
        s.index = pd.DatetimeIndex([])
        return s
    idx = s.index
    if isinstance(idx, pd.PeriodIndex):
        s.index = idx.to_timestamp().tz_localize(None)
    elif isinstance(idx, pd.DatetimeIndex):
        if idx.tz is not None:
            s.index = idx.tz_convert("UTC").tz_localize(None)
        else:
            try:
                s.index = pd.to_datetime(idx, utc=True).tz_localize(None)
            except Exception:
                s.index = pd.to_datetime(idx, errors="coerce").tz_localize(None)
    else:
        try:
            s.index = pd.to_datetime(idx, utc=True).tz_localize(None)
        except Exception:
            s.index = pd.to_datetime(idx, errors="coerce").tz_localize(None)
    return s


def load_monthly_activity(devFolder):
    """
    Total monthly activity count (commits + PRs + reviews + comments).
    Returns a tz-naive MS-freq Series.
    """
    frames = []
    for suffix, date_col in ACTIVITY_FILES:
        path = devFolder / f"{devFolder.name}{suffix}"
        if not path.exists():
            continue
        try:
            df = pd.read_excel(path, usecols=lambda c: c == date_col)
            if date_col not in df.columns:
                continue
            dt = pd.to_datetime(df[date_col], errors="coerce")
            dt = strip_tz(dt).dropna()
            if not dt.empty:
                frames.append(dt)
        except Exception:
            continue

    if not frames:
        return pd.Series(dtype=float)
    return to_monthly_series(pd.concat(frames))


def load_review_activity(devFolder):
    """Monthly review counts only."""
    path = devFolder / f"{devFolder.name}_reviews.xlsx"
    if not path.exists():
        return pd.Series(dtype=float)
    try:
        df = pd.read_excel(path)
        col = next((c for c in ["submitted_at", "created_at", "date"] if c in df.columns), None)
        if col is None:
            return pd.Series(dtype=float)
        return to_monthly_series(pd.to_datetime(df[col], errors="coerce"))
    except Exception:
        return pd.Series(dtype=float)


def load_commit_activity(devFolder):
    """Monthly commit counts only."""
    path = devFolder / f"{devFolder.name}_commits.xlsx"
    if not path.exists():
        return pd.Series(dtype=float)
    try:
        df = pd.read_excel(path)
        if "date" not in df.columns:
            return pd.Series(dtype=float)
        return to_monthly_series(pd.to_datetime(df["date"], errors="coerce"))
    except Exception:
        return pd.Series(dtype=float)


def find_risk_onsets(risk_series):
    """
    Find every month where the combined risk score crosses RISK_THRESHOLD
    and stays above it for at least 2 consecutive months — i.e. every
    burnout episode onset, not just the first.

    After each onset is flagged, an ONSET_COOLDOWN_MONTHS cooldown is
    applied before scanning resumes, so a single sustained high-risk
    stretch isn't counted as several separate onsets.

      Returns:
        chronological list of onset timestamps (empty if none found)
    """
    risk  = normalize_index(risk_series)
    above = risk >= RISK_THRESHOLD
    idx   = above.index
    n     = len(above)

    onsets = []
    i = 0
    while i < n - 1:
        if above.iloc[i] and above.iloc[i + 1]:
            onset_ts = idx[i]
            onsets.append(onset_ts)
            cooldown_until = onset_ts + pd.DateOffset(months=ONSET_COOLDOWN_MONTHS)
            j = i + 1
            while j < n and idx[j] < cooldown_until:
                j += 1
            i = j
        else:
            i += 1
    return onsets


def episode_peak_risk(risk, onset, next_onset):
    """
    Peak risk score for one episode — the window from its onset up to
    (but not including) the next episode's onset, or the cooldown boundary
    if there is no next episode.

      Returns:
        (peak_value, peak_month)
    """
    risk = normalize_index(risk)
    window_end = next_onset if next_onset is not None else (
        onset + pd.DateOffset(months=ONSET_COOLDOWN_MONTHS)
    )
    window = risk[(risk.index >= onset) & (risk.index < window_end)]
    if not window.empty:
        return float(window.max()), window.idxmax()
    return float(risk.max()), risk.idxmax()


def measure_outcomes(devFolder, onset, activity):
    """
    Given a risk onset month, measure all three outcomes from the raw activity.

      Returns:
        dict with keys: departed, activity_change, activity_declined,
        role_withdrawal, baseline_mean, post_mean
    """
    activity = normalize_index(activity)

    baseline_start = onset - pd.DateOffset(months=LOOKBACK_MONTHS)
    baseline_end   = onset
    post_start     = onset
    post_end       = onset + pd.DateOffset(months=LOOKAHEAD_MONTHS)

    baseline = activity[(activity.index >= baseline_start) & (activity.index < baseline_end)]
    post     = activity[(activity.index >= post_start)     & (activity.index < post_end)]

    baseline_mean = float(baseline.mean()) if not baseline.empty else 0.0
    post_mean     = float(post.mean())     if not post.empty     else 0.0

    # Outcome 1: departure — any 3+ consecutive silent months in post window
    departed = False
    if post.empty:
        departed = True
    else:
        full_post = pd.date_range(post_start, post_end - pd.DateOffset(months=1), freq="MS")
        post_full = post.reindex(full_post, fill_value=0)
        w = DEPARTURE_SILENCE_MONTHS
        for i in range(len(post_full) - w + 1):
            if (post_full.iloc[i: i + w] == 0).all():
                departed = True
                break

    # Outcome 2: activity decline
    if len(baseline) < MIN_BASELINE_MONTHS or post.empty or baseline_mean == 0:
        act_change   = None
        act_declined = None
    else:
        act_change   = float((post_mean - baseline_mean) / baseline_mean)
        act_declined = act_change <= -DECLINE_THRESHOLD

    # Outcome 3: role withdrawal (reviews drop, commits hold)
    commits_s = normalize_index(load_commit_activity(devFolder))
    reviews_s = normalize_index(load_review_activity(devFolder))

    def _window_mean(s, start, end):
        if s.empty:
            return 0.0
        w = s[(s.index >= start) & (s.index < end)]
        return float(w.mean()) if not w.empty else 0.0

    commit_base = _window_mean(commits_s, baseline_start, baseline_end)
    commit_post = _window_mean(commits_s, post_start, post_end)
    review_base = _window_mean(reviews_s, baseline_start, baseline_end)
    review_post = _window_mean(reviews_s, post_start, post_end)

    if commit_base == 0 or review_base == 0:
        role_withdrawal = None
    else:
        commit_change = (commit_post - commit_base) / commit_base
        review_change = (review_post - review_base) / review_base
        role_withdrawal = bool(review_change <= -0.30 and commit_change > -0.30)

    return {
        "departed":          departed,
        "activity_change":   act_change,
        "activity_declined": act_declined,
        "role_withdrawal":   role_withdrawal,
        "baseline_mean":     baseline_mean,
        "post_mean":         post_mean,
    }


def printDeveloperSummary(dev_name, outcome_df):
    """
    Print a per-episode outcome table for one developer.
    """
    print(f"\n{'=' * 78}")
    print(f"  BURNOUT EVALUATION — {dev_name}")
    print(f"  Profile: {EVAL_PROFILE}  |  Risk threshold: {RISK_THRESHOLD}  |  "
          f"Onset cooldown: {ONSET_COOLDOWN_MONTHS}mo  |  "
          f"Lookback/ahead: {LOOKBACK_MONTHS}/{LOOKAHEAD_MONTHS} months")
    print(f"{'=' * 78}")
    print(f"  {'Ep':<3} {'Onset':<9} {'PeakMonth':<11} {'PeakRisk':>9} "
          f"{'ActΔ':>8} {'Departed':>9} {'Declined':>9} {'Withdrew':>9}")
    print(f"  {'-' * 74}")

    for _, row in outcome_df.iterrows():
        ep         = str(int(row["episode"])) if pd.notna(row["episode"]) else "—"
        onset      = row["onset_month"] if pd.notna(row["onset_month"]) else "—"
        peak_month = row["peak_risk_month"] if pd.notna(row.get("peak_risk_month")) else "—"
        peak       = f"{row['peak_risk']:.3f}"
        act_d      = f"{row['activity_change']:+.0%}" if pd.notna(row["activity_change"]) else "—"
        dep        = "yes" if row["departed"]          else ("no" if row["departed"] is not None else "—")
        decl       = "yes" if row["activity_declined"] else ("no" if row["activity_declined"] is not None else "—")
        with_      = "yes" if row["role_withdrawal"]   else ("no" if row["role_withdrawal"] is not None else "—")

        print(f"  {ep:<3} {onset:<9} {peak_month:<11} {peak:>9} "
              f"{act_d:>8} {dep:>9} {decl:>9} {with_:>9}")



    print(f"{'=' * 78}\n")


def plotDeveloperTimeline(devFolder, risk, activity, onsets):
    """
    Two-panel chart for one developer:
      Top    — monthly combined risk score with threshold line, one onset
               marker per burnout episode, and a star marking the peak
               risk month within each episode
      Bottom — monthly activity count, with the year following each onset
               shaded red and the subsequent half year shaded yellow.

      Params:
        devFolder(Path): developer folder
        risk(pd.Series): monthly combined risk score
        activity(pd.Series): monthly total activity count
        onsets(list): onset timestamps from find_risk_onsets
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    dev_name = devFolder.name
    risk     = normalize_index(risk)
    activity = normalize_index(activity)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

    ax1.plot(risk.index, risk.values, color="#8e44ad", linewidth=2)
    ax1.fill_between(risk.index, 0, risk.values, alpha=0.15, color="#8e44ad")
    ax1.axhline(RISK_THRESHOLD, color="red", linestyle="--",
                linewidth=1, alpha=0.7, label=f"Risk threshold ({RISK_THRESHOLD})")
    ax1.set_ylim(0, 1.05)
    ax1.set_ylabel("Burnout Risk Score", fontsize=10)
    ax1.set_title(f"{dev_name} — burnout risk & activity timeline", fontsize=12, fontweight="bold")
    ax1.grid(True, alpha=0.25)
    ax1.legend(fontsize=9)

    ax2.bar(activity.index, activity.values, width=25,
            color="#3498db", alpha=0.6, label="Monthly activity")

    for i, onset in enumerate(onsets):
        next_onset = onsets[i + 1] if i + 1 < len(onsets) else None
        _, peak_month = episode_peak_risk(risk, onset, next_onset)

        red_end    = onset + pd.DateOffset(months=12)
        yellow_end = red_end + pd.DateOffset(months=6)

        red_label    = "Burnout window (yr 1)"    if i == 0 else None
        yellow_label = "Elevated risk (mo 13-18)" if i == 0 else None
        onset_label  = "Risk onset"               if i == 0 else None
        peak_label   = "Peak risk month"          if i == 0 else None

        ax1.axvspan(onset, red_end, alpha=0.15, color="red", label=red_label)
        ax2.axvspan(onset, red_end, alpha=0.15, color="red")

        ax1.axvspan(red_end, yellow_end, alpha=0.15, color="gold", label=yellow_label)
        ax2.axvspan(red_end, yellow_end, alpha=0.15, color="gold")

        for ax in (ax1, ax2):
            ax.axvline(onset, color="red", linestyle="-", linewidth=1.5,
                       alpha=0.8, label=onset_label)

        if peak_month in risk.index:
            ax1.scatter([peak_month], [risk.loc[peak_month]], marker="*",
                        s=180, color="#f1c40f", edgecolor="#8e5a00",
                        linewidth=0.8, zorder=5, label=peak_label)

    if onsets:
        ax1.legend(fontsize=8)



    ax2.set_ylabel("Activity count", fontsize=10)
    ax2.set_xlabel("Month", fontsize=10)
    ax2.tick_params(axis="x", rotation=40)
    ax2.grid(True, alpha=0.25)
    ax2.legend(fontsize=9)

    plt.tight_layout()
    out = devFolder / "graphs" / f"{dev_name}_burnout_timeline.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=150, bbox_inches="tight")
    print(f"  Burnout timeline saved: {out.name}")
    plt.show()
    plt.close()


def saveDeveloperOutcomes(devFolder, risk, outcome_df):
    """
    Append the monthly risk series and the per-episode outcome table to the
    developer's existing metrics workbook as two new sheets.
    """
    dev_name = devFolder.name
    metrics_path = devFolder / f"{dev_name}_metrics.xlsx"

    risk_df = pd.DataFrame({
        "month":         risk.index.strftime("%Y-%m"),
        "combined_risk": risk.values,
        "at_risk":       (risk >= RISK_THRESHOLD).values,
    })

    try:
        with pd.ExcelWriter(metrics_path, engine="openpyxl", mode="a",
                             if_sheet_exists="replace") as writer:
            risk_df.to_excel(writer, sheet_name="burnout_risk", index=False)
            outcome_df.to_excel(writer, sheet_name="burnout_episodes", index=False)
        print(f"  Burnout risk + episode sheets saved to: {metrics_path.name}")
    except Exception as e:
        fallback = devFolder / f"{dev_name}_burnout_eval.xlsx"
        with pd.ExcelWriter(fallback, engine="openpyxl") as writer:
            risk_df.to_excel(writer, sheet_name="burnout_risk", index=False)
            outcome_df.to_excel(writer, sheet_name="burnout_episodes", index=False)
        print(f"  Could not append to {metrics_path.name} ({e}); saved separately to: {fallback.name}")


def evaluateDeveloperBurnout(devFolder, profile_name=EVAL_PROFILE):
    """
    Full burnout risk + outcome evaluation for one developer. Call after
    flush_metric_data() has written the developer's metrics workbook.

      1. Build the monthly Combined Risk series from the signal sheets.
      2. Find every sustained high-risk onset (one per burnout episode).
      3. Measure departure / activity decline / role withdrawal in the
         window after each onset — skipped (with a note) if the developer
         was never meaningfully active overall.
      4. Print the summary, plot the risk + activity timeline, and save
         both to the metrics workbook.

      Params:
        devFolder(Path): developer folder
        profile_name(string): which WEIGHT_PROFILES entry to score with

      Returns:
        dict with the developer's monthly risk series ("risk") and a
        DataFrame of per-episode outcomes ("outcomes"), or None if there
        wasn't enough signal data to score
    """
    dev_name = devFolder.name
    profile  = WEIGHT_PROFILES[profile_name]

    print(f"\nRunning burnout evaluation for {dev_name} (profile: {profile_name})...")

    risk = score_developer_full(devFolder, profile)
    if risk is None or len(risk) < 4:
        print(f"  Not enough monthly signal data to score {dev_name}.")
        return None

    activity = load_monthly_activity(devFolder)
    overall_mean = float(activity.mean()) if not activity.empty else 0.0
    if activity.empty:
        print(f"  No raw activity files found for {dev_name} — outcome measurement skipped.")

    onsets = find_risk_onsets(risk)
    skip_outcomes = overall_mean < MIN_BASELINE_ACTIVITY
    if onsets and skip_outcomes:
        print(f"  Average activity ({overall_mean:.1f}/month) is below the "
              f"{MIN_BASELINE_ACTIVITY} floor — outcome measurement skipped, "
              f"risk is still reported below.")

    rows = []
    if not onsets:
        print("  No sustained high-risk period found.")
        peak_val, peak_month = float(risk.max()), risk.idxmax()
        rows.append({
            "developer": dev_name, "episode": None, "onset_month": None,
            "peak_risk": peak_val, "peak_risk_month": peak_month.strftime("%Y-%m"),
            "mean_risk": float(risk.mean()), "baseline_activity": overall_mean,
            "ever_high_risk": False, "departed": None, "activity_change": None,
            "activity_declined": None, "role_withdrawal": None,
            "baseline_mean": None, "post_mean": None,
        })
    else:
        for ep_num, onset in enumerate(onsets, start=1):
            next_onset = onsets[ep_num] if ep_num < len(onsets) else None
            peak_val, peak_month = episode_peak_risk(risk, onset, next_onset)
            print(f"  Episode #{ep_num}: onset {onset.strftime('%Y-%m')}  "
                  f"(peak risk {peak_val:.3f} in {peak_month.strftime('%Y-%m')})")

            if skip_outcomes:
                outcomes = {"departed": None, "activity_change": None,
                            "activity_declined": None, "role_withdrawal": None,
                            "baseline_mean": None, "post_mean": None}
            else:
                outcomes = measure_outcomes(devFolder, onset, activity)

            rows.append({
                "developer": dev_name, "episode": ep_num,
                "onset_month": onset.strftime("%Y-%m"),
                "peak_risk": peak_val, "peak_risk_month": peak_month.strftime("%Y-%m"),
                "mean_risk": float(risk.mean()), "baseline_activity": overall_mean,
                "ever_high_risk": True, **outcomes,
            })

    outcome_df = pd.DataFrame(rows)
    printDeveloperSummary(dev_name, outcome_df)
    plotDeveloperTimeline(devFolder, risk, activity, onsets)
    saveDeveloperOutcomes(devFolder, risk, outcome_df)

    return {"risk": risk, "outcomes": outcome_df}


## MAIN LOOP
Mounts Drive, prompts for a developer, runs every signal cell above, then
runs the burnout calculation for them.

In [ ]:
# MAIN LOOP

dev_name = input("Enter Developer Name: ")
dev_path = DEVPATH / dev_name

if not dev_path.exists():
    print(f"Developer directory '{dev_name}' not found")
else:
    print(f"\n{'=' * 60}")
    print(f"Processing {dev_path.name}...")
    print(f"{'=' * 60}")

    devMessages = load_messages(dev_path)
    if devMessages.empty:
        print(f"  No messages found for {dev_path.name}, aborting.")
    else:
        arousal(dev_path, devMessages, vad_dict)
        commits(dev_path)
        pronouns(dev_path, devMessages)
        lexicalDiversity(dev_path, devMessages)
        LF(dev_path, devMessages)
        off_hours(dev_path)
        PRThroughput(dev_path)
        task_abandonment(dev_path)
        compute_sentiment(dev_path)

        flush_metric_data(dev_path)

        evaluateDeveloperBurnout(dev_path)
        print(f"\n {dev_path.name} complete.")
